In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta") \
               .load("/Volumes/workspace/bronze/bronzevolume/bookings/data/")

df.display()

In [0]:
df = df.withColumn("amount", col("amount").cast(DoubleType())) \
        .withColumn("modified_date", current_timestamp()) \
        .withColumn("booking_date", to_date(col("booking_date"))) \
        .drop("_rescued_data")

df.display()

In [0]:
import dlt

In [0]:
@dlt.table (
    name = "bookings"
)

# load the table incrementally
def stage_bookings():
    df = spark.readStream.format("delta") \
               .load("/Volumes/workspace/bronze/bronzevolume/bookings/data/")
    return df


In [0]:
@dlt.view(
    name = "trans_bookings"
)

def trans_bookings():
    df = spark.readStream.table("stage_bookings")
    df.withColumn("amount", col("amount").cast(DoubleType())) \
        .withColumn("modified_date", current_timestamp()) \
        .withColumn("booking_date", to_date(col("booking_date"))) \
        .drop("_rescued_data")
    return df


In [0]:
# rules for checking/validating data
rules = {
    "rule1" : "booking_id IS NOT NULL",
    "rule2" : "passenger_id IS NOT NULL"
}

In [0]:

@dlt.table(
    name = "silver_bookings"
)
@dlt.expect_all_or_drop(rules)   # dlt.expect_all results in warn, fail, or drop
def silver_bookings():
    df = spark.readStream.table("trans_bookings")
    return df